<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Classification with Spark MLlib (German Credit)

*Session 5 · Notebook 03.05 · Practical · Student version*

## About this notebook

The lecture (notebook 03.04) used the Titanic passengers. Here you apply the same three models to a **real credit problem**, the German credit data from Session 4 (notebook 03.03), in the same order as the lecture:

- **Part A:** load, explore and split the data.
- **Part B:** a decision tree and a random forest.
- **Part C:** a logistic regression, compared with the trees.
- **Part D (capstone):** the data has **class imbalance** (70% good loans, 30% bad), so a model can look accurate while missing most of the bad loans. You fix that by moving the decision threshold, then recommend a model.
- **Challenge:** rebuild the decision tree as one pipeline.

## About the exercises

Each task appears as two cells:

1. A markdown cell describing what to do (sometimes with a `> *Hint:*`).
2. A `# Your turn` cell for you to write your answer.

This is the student version, so the answers are not included. Work top to bottom: later tasks reuse variables from earlier answers.

## About the data

**German credit** (`german_credit`): 1,000 past loan applicants with 20 features (13 categorical such as `checking_status`, `credit_history`, `purpose`; 7 numeric such as `duration_months`, `credit_amount`, `age_years`) and the target `risk` (`good` 700, `bad` 300). **`bad` is the positive class (1)**: it is the event the model must catch.

## Index

- [Part 0: Setup and helpers](#p0)
- [Part A: Load, explore and split](#pa)
- [Part B: Decision tree and random forest](#pb)
- [Part C: Logistic regression](#pc)
- [Part D: Capstone: fixing the imbalance](#pd)
- [Challenge: the decision tree as one pipeline](#challenge)
- [Further practice](#further)

<a id="p0"></a>
# Part 0: Setup and helpers

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Imputer
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
SEED = 42
from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_03_05_Practical_Classification").getOrCreate()
print("Spark version:", spark.version)

The helpers from the lecture: feature names from a vector column, a confusion matrix, and the classification metrics for a chosen positive class.

In [ ]:
def feature_names(df, col="features"):
    attrs = df.schema[col].metadata["ml_attr"]["attrs"]
    return [a["name"] for a in sorted((a for g in attrs.values() for a in g), key=lambda a: a["idx"])]

def confusion(pred, label):
    return (pred.groupBy(label).pivot("prediction", [0.0, 1.0]).count()
                .na.fill(0).orderBy(label).withColumnRenamed(label, "actual"))

def scores(pred, label, name="model"):
    mc = lambda metric, **kw: MulticlassClassificationEvaluator(labelCol=label, metricName=metric, **kw).evaluate(pred)
    return {"model": name,
            "accuracy": round(mc("accuracy"), 3),
            "precision_1": round(mc("precisionByLabel", metricLabel=1.0), 3),
            "recall_1": round(mc("recallByLabel", metricLabel=1.0), 3),
            "auc": round(BinaryClassificationEvaluator(labelCol=label, rawPredictionCol="probability").evaluate(pred), 3)}  # probability: right for trees too

p1 = vector_to_array("probability")[1]     # probability of the positive class, as a number

<a id="pa"></a>
# Part A: Load, explore and split

### Exercise A1: Load and inspect

Read `german_credit` into `credit`, print its schema, the class balance of `risk` and the bad rate. Add a numeric label `label` = 1.0 for `bad` and 0.0 for `good`.

> *Hint:* An explicit `F.when` makes the positive class unmistakable; a `StringIndexer` would choose by frequency.

In [ ]:
# Your turn

### Exercise A2: Explore credit amount and purpose

Compare the average and median `credit_amount` for good and bad loans, and show the bad rate by `purpose`, sorted from worst to best (only purposes with at least 20 loans).

In [ ]:
# Your turn

### Exercise A3: Split with the same bad rate

Split `credit` 70/30 with `seed=SEED` into `c_train` and `c_test`, and confirm the bad rate is similar in both. (`randomSplit` is not stratified; with 1,000 rows the rates should still be close.)

In [ ]:
# Your turn

<a id="pb"></a>
# Part B: Decision tree and random forest

**Shared preparation (done for you).** As in Part A of the lecture, the trees only need two steps: give each category a number (`StringIndexer`), then pack every column into one `features` vector (`VectorAssembler`). Each step is learned from `c_train` and applied to both parts, so the tree and the forest below are trained on ready-made data.

In [ ]:
# The column types: 13 text columns and 7 numeric columns (the label is not a feature)
text_cols = [c for c, t in credit.dtypes if t == "string" and c != "risk"]
num_cols = [c for c, t in credit.dtypes if t in ("int", "bigint", "double") and c != "label"]
idx_cols = [c + "_idx" for c in text_cols]            # checking_status -> checking_status_idx, ...
print(len(text_cols), "categorical and", len(num_cols), "numeric features")

# Step 1: number the categories, learned from the TRAINING data only
indexer = StringIndexer(inputCols=text_cols, outputCols=idx_cols, handleInvalid="keep")
indexer_model = indexer.fit(c_train)

# Step 2: pack the numeric columns and the category numbers into one vector
credit_features = num_cols + idx_cols
assembler = VectorAssembler(inputCols=credit_features, outputCol="features")

train_tree = assembler.transform(indexer_model.transform(c_train))
test_tree = assembler.transform(indexer_model.transform(c_test))
train_tree.select("checking_status", "checking_status_idx", "features", "label").show(3, truncate=60)

### Exercise B1: A single decision tree

Fit a `DecisionTreeClassifier(labelCol="label", maxDepth=6, seed=SEED)` on `train_tree` as `dtree`. Score `test_tree`, show the confusion matrix and the metrics. Pay attention to **recall on bad loans**: of all truly bad loans, how many did the tree catch?

In [ ]:
# Your turn

### Exercise B2: A random forest

Fit a `RandomForestClassifier(labelCol="label", numTrees=200, maxDepth=8, seed=SEED)` on `train_tree` as `rfc`. Score `test_tree` and print the confusion matrix and metrics. **Do you notice anything strange?** Compare its recall on bad loans with its accuracy.

In [ ]:
# Your turn

### Exercise B3: Which model is better?

Put both score dictionaries in one pandas table. Compare accuracy with recall on bad loans. Why is accuracy the wrong headline number here? (Hint: what accuracy would a model that always says "good" get?)

In [ ]:
# Your turn

### Exercise B4: Feature importances

Plot the ten most important features of the random forest. Which matter most? Does that match credit intuition?

In [ ]:
# Your turn

<a id="pc"></a>
# Part C: Logistic regression

As in the lecture (Section 8), logistic regression multiplies each input by a coefficient, so the category numbers are not enough: the 13 text columns must be **one-hot encoded**. The baseline of each column is its dropped, least frequent category.

### Exercise C1: One-hot encode and fit

Build a pipeline: a `StringIndexer` for `text_cols` (to `<col>_idx`), a `OneHotEncoder` (to `<col>_ohe`), a `VectorAssembler` of `num_cols` plus the one-hot columns into `features`, and `LogisticRegression(labelCol="label", maxIter=200)`. Fit it on `c_train` as `logreg`, score `c_test` as `lr_pred`, show the confusion matrix and the metrics.

> *Hint:* The steps of the lecture's Part B, inside one `Pipeline`: it is fitted on the raw `c_train` and scores the raw `c_test`.

In [ ]:
# Your turn

### Exercise C2: Compare with the trees

Put the decision tree, random forest and logistic regression scores in one table. Then plot the ROC curves of the forest and the logistic regression together (bring the label and probability to pandas, as in the lecture).

In [ ]:
# Your turn

### Exercise C3: The riskiest applicants

Using `lr_pred`, add the probability of a bad loan as a column `p_bad` (rounded to 3 decimals). Show the ten test applicants with the **highest** `p_bad`, with `purpose`, `credit_amount`, `duration_months`, `risk` and `p_bad`. How many of them were actually bad loans?

> *Hint:* `p1` (Part 0) is the probability of the positive class as a number; then `orderBy(F.desc(...))` and `limit(10)`.

In [ ]:
# Your turn

<a id="pd"></a>
# Part D: Capstone: fixing the imbalance

The simplest way to make a model catch more of the rare class is to **move the threshold** (lecture Section 10): label a loan `bad` at a lower probability than 0.5.

### Capstone part 1: Move the threshold

Using the forest's probability of `bad`, relabel the test predictions at thresholds 0.5, 0.4 and 0.3, and print recall and precision on bad loans and accuracy for each. How much does recall improve at 0.3, and what does it cost?

In [ ]:
# Your turn

### Capstone part 2: Recommendation

Collect every model and variant you built into one table (AUC, accuracy, precision and recall on bad). Which would you recommend to a credit risk team, and at what threshold? Justify it in two or three sentences.

In [ ]:
# Your turn

<a id="challenge"></a>
# Challenge: the decision tree as one pipeline

In Part B you prepared the data step by step and trained the tree on the result. In Part C the logistic regression did the same work inside one `Pipeline`. Do that for the tree too.

### Challenge: Build the tree pipeline

Build a `Pipeline` with three stages: the `StringIndexer` and the `VectorAssembler` from Part B, then `DecisionTreeClassifier(labelCol="label", maxDepth=6, seed=SEED)`. Fit it on the **raw** `c_train`, score the raw `c_test`, and compare the metrics with `dtree_scores` from Exercise B1. Why might a pipeline be the better way to keep the model?

> *Hint:* Pass the raw DataFrames: the pipeline does the indexing and assembling itself.

In [ ]:
# Your turn

<a id="further"></a>
## Further practice

- Choose the forest's `maxDepth` with a **validation split** of `c_train` (as in practical 03.03, Exercise 5.3), comparing **area under the precision-recall curve** (`metricName="areaUnderPR"`), which focuses on the rare class.
- Compare these Spark results with your Session 4 scikit-learn results for the same data: same story, different splits.